# TripMe Part 10 — Second Corrective QLoRA Retraining
Attach `tripme-part10-corrective-train`, enable a T4 GPU, keep Internet on, and Run All. This retrains from the base model on the 2,338/340 combined corrective dataset.

In [ ]:
# Preserve Kaggle's datasets/fsspec/dill packages to avoid dependency conflicts.
!pip install -q --no-cache-dir transformers==4.48.3 accelerate==1.3.0 peft==0.14.0 bitsandbytes==0.48.2 safetensors>=0.4
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import torch, bitsandbytes as bnb
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'bitsandbytes:', bnb.__version__)
assert torch.cuda.is_available(), 'GPU is not enabled'
assert torch.cuda.device_count() == 1

In [ ]:
from pathlib import Path
import json, math, random, shutil
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
MAX_LENGTH = 768
MAX_NEW_TOKENS = 384
EPOCHS = 1
SEED = 44
matches = list(Path('/kaggle/input').rglob('combined_train.jsonl'))
if not matches:
    raise FileNotFoundError('Attach tripme-part10-corrective-train')
DATA_DIR = matches[0].parent
RUN_DIR = Path('/kaggle/working/tripme-second-corrective-run')
OUTPUT_DIR = Path('/kaggle/working/tripme-second-corrective-adapter')
print('GPU:', torch.cuda.get_device_name(0), 'Data:', DATA_DIR)

In [ ]:
from datasets import load_dataset
data = load_dataset('json', data_files={
    'train': str(DATA_DIR/'combined_train.jsonl'),
    'validation': str(DATA_DIR/'combined_validation.jsonl'),
})
assert len(data['train']) == 2338
assert len(data['validation']) == 340
print(data)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = 'right'
quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant, device_map={'': 0}, torch_dtype=torch.float16
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.08, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
def tokenize_completion(example):
    full_text = tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)
    prompt_text = tokenizer.apply_chat_template(example['messages'][:-1], tokenize=False, add_generation_prompt=True)
    full = tokenizer(full_text, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False)
    prompt = tokenizer(prompt_text, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False)
    labels = full['input_ids'].copy()
    prompt_length = min(len(prompt['input_ids']), len(labels))
    labels[:prompt_length] = [-100] * prompt_length
    full['labels'] = labels
    return full
tokenized = data.map(tokenize_completion, remove_columns=data['train'].column_names)
assert all(any(value != -100 for value in row['labels']) for row in tokenized['train'].select(range(20)))
print('Tokenization passed')

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
args = TrainingArguments(
    output_dir=str(RUN_DIR), num_train_epochs=EPOCHS,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, learning_rate=7e-5,
    warmup_ratio=0.08, lr_scheduler_type='cosine', weight_decay=0.02,
    logging_steps=10, eval_strategy='steps', eval_steps=100,
    save_strategy='steps', save_steps=100, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    fp16=True, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit', report_to='none', seed=SEED, data_seed=SEED,
)
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, padding=True, label_pad_token_id=-100, return_tensors='pt'
)
trainer = Trainer(
    model=model, args=args, train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'], data_collator=collator,
)
result = trainer.train()
evaluation = trainer.evaluate()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

In [ ]:
# Generate targeted Sinhala failure-scenario samples with a larger output limit.
model.config.use_cache = True
model.eval()
target_scenarios = {'current_fact_refusal', 'budget_uncertainty', 'current_uncertainty', 'family_accessibility', 'culture_etiquette', 'itinerary_place_retention', 'comparison'}
candidates = [row for row in data['validation'] if row['lang'] == 'si' and row['scenario'] in target_scenarios]
random.Random(SEED).shuffle(candidates)
samples = candidates[:20]
generations = []
for row in samples:
    prompt_messages = row['messages'][:-1]
    inputs = tokenizer.apply_chat_template(
        prompt_messages, add_generation_prompt=True, return_tensors='pt'
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs.shape[-1]:]
    generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    generations.append({
        'lang': row['lang'], 'scenario': row['scenario'], 'place_ids': row['place_ids'],
        'prompt_messages': prompt_messages, 'reference': row['messages'][-1]['content'],
        'generated': generated, 'generated_tokens': int(new_tokens.shape[-1]),
        'hit_token_limit': int(new_tokens.shape[-1]) >= MAX_NEW_TOKENS,
    })
with (OUTPUT_DIR/'targeted_validation_generations.jsonl').open('w', encoding='utf-8') as handle:
    for row in generations:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')
print('Saved targeted generations:', len(generations))

In [ ]:
metrics = {**result.metrics, **{f'final_{key}': value for key, value in evaluation.items()}}
if 'final_eval_loss' in metrics and metrics['final_eval_loss'] < 50:
    metrics['final_perplexity'] = math.exp(metrics['final_eval_loss'])
manifest = {
    'status': 'second_corrective_training_complete', 'model_id': MODEL_ID,
    'train_rows': len(data['train']), 'validation_rows': len(data['validation']),
    'epochs': EPOCHS, 'max_length': MAX_LENGTH, 'max_new_tokens': MAX_NEW_TOKENS,
    'seed': SEED, 'training_origin': 'base_model_not_previous_adapter',
    'dataset_status': 'ai_corrective_draft_requires_human_review',
}
(OUTPUT_DIR/'training_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(OUTPUT_DIR/'training_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = shutil.make_archive('/kaggle/working/tripme_part10_corrective_output', 'zip', OUTPUT_DIR)
print(json.dumps({'manifest': manifest, 'metrics': metrics}, indent=2))
print('Download:', archive)

In [ ]:
from IPython.display import HTML, display
zip_path = Path('/kaggle/working/tripme_part10_corrective_output.zip')
assert zip_path.is_file()
print('Exists:', zip_path.exists(), 'Size MB:', round(zip_path.stat().st_size/1024/1024, 2))
display(HTML("<a href='files/tripme_part10_corrective_output.zip' download>Download tripme_part10_corrective_output.zip</a>"))